# Customer Churn Prediction

**Goal:** predict which customers are about to stop buying, so a business knows who to contact first.

**Data:** two years of sales from a UK online retailer (December 2009 to December 2011), about 804,000 transactions and 5,855 customers.

**This notebook has two parts:**

- **Part 1 (Steps 1 to 10): first approach.** A snapshot model that identifies customers who have already gone quiet. Compares four algorithms, ensembles and feature sets.

- **Part 2 (Steps 11 to 17): predicting who will leave next.** A forward-looking model, checks that the result is real, the result in numbers of customers, a contact list for the next 90 days, MLflow tracking, and the API and live app.

---

# Part 1: First Approach (Snapshot Model)

## Step 1: Load the Data

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("cleaned_retail.csv")
rfm = pd.read_csv("rfm_table.csv")

data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

print(data.shape)
print(rfm.shape)
print(rfm.head())

(804824, 8)
(5855, 4)
   Customer ID  Recency  Frequency  Monetary
0      15712.0        9         12   3472.41
1      12931.0       21         57  92347.34
2      17611.0        7         28   8321.98
3      16620.0        3          6   1758.93
4      13381.0       31         12   8224.37


## Step 2: Define the Churn Label

In [3]:
rfm["Churned"] = (rfm["Recency"] > 90).astype(int)

print(rfm["Churned"].value_counts())
print(rfm["Churned"].value_counts(normalize=True) * 100)


Churned
1    2968
0    2887
Name: count, dtype: int64
Churned
1    50.691716
0    49.308284
Name: proportion, dtype: float64


## Churn Label Definition

Defined churn as: no purchase within the last 90 days, based on the Recency feature from the RFM table (built in my earlier segmentation project). This threshold was chosen deliberately — it sits close to the dataset's median Recency (95 days), producing a genuinely balanced target (50.7% churned, 49.3% active) rather than a heavily skewed one. A shorter threshold (e.g., 30 days) would have classified most customers as "churned," making the label far less useful for a real business decision.

## Step 3: Train/Test Split (with Leakage Check)

Excluded Recency from the feature set, since it was used to define the Churned label itself (Recency > 90 = Churned) — including it would let the model simply learn back the labeling rule rather than genuinely predict churn. Used a stratified split to preserve the ~50/50 churn balance in both train and test sets.

In [4]:
from sklearn.model_selection import train_test_split

feature_cols = ["Frequency", "Monetary"]
X = rfm[feature_cols]
y = rfm["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())


Train: (4684, 2) Test: (1171, 2)
Train churn rate: 0.5068317677198976
Test churn rate: 0.5072587532023911


## Step 4: Baseline Model — Logistic Regression

Building a simple baseline first, before the more complex model, to establish a minimum performance bar. Logistic Regression is a simple, well understood model, which makes it a fair starting point.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_preds))
print("Precision:", precision_score(y_test, baseline_preds))
print("Recall:", recall_score(y_test, baseline_preds))
print("F1:", f1_score(y_test, baseline_preds))

Accuracy: 0.6900085397096499
Precision: 0.6542056074766355
Recall: 0.8249158249158249
F1: 0.7297096053611318


## Baseline Results

Logistic Regression on Frequency + Monetary alone: Accuracy 69.0%, Precision 65.4%, Recall 82.5%, F1 0.73. The model is biased toward flagging customers as churned (higher recall than precision) — a reasonable tradeoff for this use case, since missing an actual at-risk customer is typically costlier than a false alarm.

## Step 5: XGBoost Model

Building the primary model — XGBoost, a gradient-boosted tree ensemble — to see if it captures more signal than the linear baseline. Same features, same train/test split, for a fair comparison.

In [6]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Precision:", precision_score(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds))
print("F1:", f1_score(y_test, xgb_preds))

Accuracy: 0.6942783945345858
Precision: 0.6755952380952381
Recall: 0.7643097643097643
F1: 0.717219589257504


## Step 6: Feature Engineering — New Features & Interactions

Adding features beyond the original Frequency/Monetary, without reintroducing Recency leakage:
- **AvgOrderValue** (Monetary / Frequency) — spend per transaction, a genuinely different signal than either raw feature alone
- **Frequency_log, Monetary_log** — log-transformed versions, since both are right-skewed (seen in my earlier segmentation project)
- **Tenure** — days between a customer's first and last purchase, computed from the raw transaction data. This reflects how long someone has been a customer, distinct from Recency (which reflects how recently they last bought) — not leakage, since it doesn't encode "distance from the dataset's end date."
- **Frequency × Monetary interaction** — lets the model see combined high-frequency/high-spend behavior directly, rather than requiring it to learn the interaction itself

In [7]:
import numpy as np

# Tenure: days between first and last purchase, per customer
tenure = data.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
tenure["Tenure"] = (tenure["max"] - tenure["min"]).dt.days
tenure = tenure[["Tenure"]].reset_index()

rfm = rfm.merge(tenure, on="Customer ID", how="left")

rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["Frequency_log"] = np.log1p(rfm["Frequency"])
rfm["Monetary_log"] = np.log1p(rfm["Monetary"])
rfm["Freq_Monetary_Interaction"] = rfm["Frequency"] * rfm["Monetary"]

print(
    rfm[
        [
            "Customer ID",
            "Frequency",
            "Monetary",
            "Tenure",
            "AvgOrderValue",
            "Frequency_log",
            "Monetary_log",
            "Freq_Monetary_Interaction",
        ]
    ].head()
)


   Customer ID  Frequency  Monetary  Tenure  AvgOrderValue  Frequency_log  \
0      15712.0         12   3472.41     729     289.367500       2.564949   
1      12931.0         57  92347.34     717    1620.128772       4.060443   
2      17611.0         28   8321.98     731     297.213571       3.367296   
3      16620.0          6   1758.93     734     293.155000       1.945910   
4      13381.0         12   8224.37     706     685.364167       2.564949   

   Monetary_log  Freq_Monetary_Interaction  
0      8.152892                   41668.92  
1     11.433323                 5263798.38  
2      9.026776                  233015.44  
3      7.473029                   10553.58  
4      9.014979                   98692.44  


## Step 7: Rebuild Train/Test Split with Expanded Features

In [8]:
feature_cols_v2 = [
    "Frequency",
    "Monetary",
    "Tenure",
    "AvgOrderValue",
    "Frequency_log",
    "Monetary_log",
    "Freq_Monetary_Interaction",
]

X2 = rfm[feature_cols_v2]
y2 = rfm["Churned"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print("Train:", X2_train.shape, "Test:", X2_test.shape)


Train: (4684, 7) Test: (1171, 7)


## Step 8: Three-Model Comparison on Expanded Features

Re-running Logistic Regression and XGBoost on the new 7-feature set, plus adding a Multi-Layer Perceptron (MLPClassifier) — a genuine neural network — using scikit-learn's implementation rather than TensorFlow, which keeps the environment simple.

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Scale features - neural networks and logistic regression both benefit from this,
# especially now that Freq_Monetary_Interaction has a very different scale than the rest
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

# Logistic Regression (v2)
lr_v2 = LogisticRegression(max_iter=1000, random_state=42)
lr_v2.fit(X2_train_scaled, y2_train)
lr_v2_preds = lr_v2.predict(X2_test_scaled)

# XGBoost (v2) - tree-based, doesn't need scaling, use original unscaled features
xgb_v2 = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_v2.fit(X2_train, y2_train)
xgb_v2_preds = xgb_v2.predict(X2_test)

# Neural Network (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42)
mlp.fit(X2_train_scaled, y2_train)
mlp_preds = mlp.predict(X2_test_scaled)

for name, preds in [
    ("Logistic Regression v2", lr_v2_preds),
    ("XGBoost v2", xgb_v2_preds),
    ("Neural Network (MLP)", mlp_preds),
]:
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y2_test, preds))
    print("Precision:", precision_score(y2_test, preds))
    print("Recall:", recall_score(y2_test, preds))
    print("F1:", f1_score(y2_test, preds))



Logistic Regression v2
Accuracy: 0.7096498719043552
Precision: 0.699685534591195
Recall: 0.7491582491582491
F1: 0.7235772357723578

XGBoost v2
Accuracy: 0.7190435525192144
Precision: 0.6939970717423133
Recall: 0.797979797979798
F1: 0.7423649177760376

Neural Network (MLP)
Accuracy: 0.7216054654141759
Precision: 0.6942028985507246
Recall: 0.8063973063973064
F1: 0.7461059190031153


## Step 9: Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_v2 = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_v2.fit(X2_train, y2_train)  # tree-based, no scaling needed
rf_v2_preds = rf_v2.predict(X2_test)

print("Accuracy:", accuracy_score(y2_test, rf_v2_preds))
print("Precision:", precision_score(y2_test, rf_v2_preds))
print("Recall:", recall_score(y2_test, rf_v2_preds))
print("F1:", f1_score(y2_test, rf_v2_preds))


Accuracy: 0.7241673783091375
Precision: 0.6960926193921853
Recall: 0.8097643097643098
F1: 0.7486381322957198


## Step 10: Ensemble

Rebuilt the ensemble to include all four base models — Logistic Regression, XGBoost, Neural Network, and Random Forest — combined via soft voting.

In [11]:
from sklearn.ensemble import VotingClassifier

ensemble_v2 = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=42)),
        (
            "xgb",
            XGBClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
            ),
        ),
        (
            "mlp",
            MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        ),
        ("rf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
    ],
    voting="soft",
)

ensemble_v2.fit(X2_train_scaled, y2_train)
ensemble_v2_preds = ensemble_v2.predict(X2_test_scaled)

print("Accuracy:", accuracy_score(y2_test, ensemble_v2_preds))
print("Precision:", precision_score(y2_test, ensemble_v2_preds))
print("Recall:", recall_score(y2_test, ensemble_v2_preds))
print("F1:", f1_score(y2_test, ensemble_v2_preds))


Accuracy: 0.7275832621690862
Precision: 0.7055306427503737
Recall: 0.7946127946127947
F1: 0.7474267616785432


## Final Model Selection

Adding Random Forest to the ensemble (4 models total) did not meaningfully improve over the 3-model ensemble or standalone Random Forest — Accuracy remained at 72.8%, F1 (0.747) sat between the two comparison points rather than exceeding either. This suggests Random Forest's predictions substantially overlap with the existing ensemble members, adding redundancy rather than new signal.

**Final decision:** given the standalone **Random Forest model has the best F1 (0.749) of any single model, with 81.0% recall (only the 2-feature logistic baseline recalled slightly more, 82.5%, at much lower precision), and is simpler to deploy than a 4-model ensemble**, Random Forest is selected as the best model of this first approach — trading a marginal amount of ensemble complexity for near-identical performance and a much simpler deployment footprint.

<br>

---

# Part 2: Predicting Who Will Leave Next

Everything above uses one snapshot of the data. The churn label and the features come from the same two-year window, so the model finds customers who have **already** gone quiet.

A business needs to know who **will** stop buying. Part 2 rebuilds the problem to look forward, using the same features and the same four models.

<br>

<br>

## Step 11: Forward-Looking Data

Pick a cutoff date and only let the model see what happened before it.

- **Cutoff date:** 90 days before the data ends

- **Features:** built only from purchases **before** the cutoff

- **Label:** Churned = no purchase in the **90 days after** the cutoff

<br>

In [ ]:
# The cutoff date: 90 days before the data ends
cutoff = data["InvoiceDate"].max() - pd.Timedelta(days=90)

# Split the transactions into before and after the cutoff
before = data[data["InvoiceDate"] <= cutoff].copy()
after = data[data["InvoiceDate"] > cutoff]

print("Cutoff date:", cutoff.date())
print("Transactions before the cutoff:", len(before))
print("Transactions after the cutoff:", len(after))

Cutoff date: 2011-09-10
Transactions before the cutoff: 643403
Transactions after the cutoff: 161421


<br>

Build one row per customer from the transactions before the cutoff.

Recency is added as a feature. It is fair to use here because it is measured at the cutoff, before the 90 days being predicted.

<br>

In [ ]:
# One row per customer, using only the transactions before the cutoff
before["Amount"] = before["Quantity"] * before["Price"]

fwd = before.groupby("Customer ID").agg(
    Frequency=("Invoice", "nunique"),
    Monetary=("Amount", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
)

# Same engineered features as Step 6
fwd["Tenure"] = (fwd["last_purchase"] - fwd["first_purchase"]).dt.days
fwd["AvgOrderValue"] = fwd["Monetary"] / fwd["Frequency"]
fwd["Frequency_log"] = np.log1p(fwd["Frequency"])
fwd["Monetary_log"] = np.log1p(fwd["Monetary"].clip(lower=0))
fwd["Freq_Monetary_Interaction"] = fwd["Frequency"] * fwd["Monetary"]

# New feature: days since the last purchase, measured at the cutoff
fwd["Recency"] = (cutoff - fwd["last_purchase"]).dt.days

print(fwd[["Frequency", "Monetary", "Tenure", "Recency"]].head())

             Frequency  Monetary  Tenure  Recency
Customer ID                                      
12346.0              4  77353.96     395      235
12347.0              6   4114.18     274       39
12348.0              4   1709.40     189      158
12349.0              3   2671.14     181      317
12350.0              1    334.40       0      219


<br>

Create the label from the transactions after the cutoff.

<br>

In [ ]:
# Label: a customer has churned if they bought nothing after the cutoff
bought_again = after["Customer ID"].unique()
fwd["Churned"] = (~fwd.index.isin(bought_again)).astype(int)

print("Customers:", len(fwd))
print(fwd["Churned"].value_counts())
print(fwd["Churned"].value_counts(normalize=True) * 100)

Customers: 5258
Churned
1    2968
0    2290
Name: count, dtype: int64
Churned
1    56.447318
0    43.552682
Name: proportion, dtype: float64


<br>

### Forward-Looking Dataset

- **5,258 customers** bought at least once before the cutoff (2011-09-10).

- **2,968 of them (56.4%)** did not buy again in the next 90 days.

<br>

<br>

---

## Step 12: Model Comparison on the Forward-Looking Data

Same train/test split as before: 80% of customers to train, 20% held out to test.

<br>

In [ ]:
feature_cols_fwd = [
    "Frequency",
    "Monetary",
    "Tenure",
    "AvgOrderValue",
    "Frequency_log",
    "Monetary_log",
    "Freq_Monetary_Interaction",
    "Recency",
]

Xf = fwd[feature_cols_fwd]
yf = fwd["Churned"]

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    Xf, yf, test_size=0.2, random_state=42, stratify=yf
)

print("Train:", Xf_train.shape, "Test:", Xf_test.shape)

Train: (4206, 8) Test: (1052, 8)


<br>

Train the same four models as in Steps 8 and 9, and score them on the test customers.

<br>

In [ ]:
from sklearn.metrics import roc_auc_score

# Scale features for Logistic Regression and the Neural Network
scaler_fwd = StandardScaler()
Xf_train_scaled = scaler_fwd.fit_transform(Xf_train)
Xf_test_scaled = scaler_fwd.transform(Xf_test)

# Logistic Regression
lr_fwd = LogisticRegression(max_iter=1000, random_state=42)
lr_fwd.fit(Xf_train_scaled, yf_train)
lr_fwd_preds = lr_fwd.predict(Xf_test_scaled)
lr_fwd_probs = lr_fwd.predict_proba(Xf_test_scaled)[:, 1]

# XGBoost
xgb_fwd = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_fwd.fit(Xf_train, yf_train)
xgb_fwd_preds = xgb_fwd.predict(Xf_test)
xgb_fwd_probs = xgb_fwd.predict_proba(Xf_test)[:, 1]

# Neural Network (MLP)
mlp_fwd = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42)
mlp_fwd.fit(Xf_train_scaled, yf_train)
mlp_fwd_preds = mlp_fwd.predict(Xf_test_scaled)
mlp_fwd_probs = mlp_fwd.predict_proba(Xf_test_scaled)[:, 1]

# Random Forest
rf_fwd = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_fwd.fit(Xf_train, yf_train)
rf_fwd_preds = rf_fwd.predict(Xf_test)
rf_fwd_probs = rf_fwd.predict_proba(Xf_test)[:, 1]

for name, preds, probs in [
    ("Logistic Regression", lr_fwd_preds, lr_fwd_probs),
    ("XGBoost", xgb_fwd_preds, xgb_fwd_probs),
    ("Neural Network (MLP)", mlp_fwd_preds, mlp_fwd_probs),
    ("Random Forest", rf_fwd_preds, rf_fwd_probs),
]:
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(yf_test, preds))
    print("Precision:", precision_score(yf_test, preds))
    print("Recall:", recall_score(yf_test, preds))
    print("F1:", f1_score(yf_test, preds))
    print("AUC:", roc_auc_score(yf_test, probs))


Logistic Regression
Accuracy: 0.7395437262357415
Precision: 0.7531645569620253
Recall: 0.8013468013468014
F1: 0.7765089722675367
AUC: 0.8090107773513887

XGBoost
Accuracy: 0.7557034220532319
Precision: 0.7549167927382754
Recall: 0.8400673400673401
F1: 0.7952191235059761
AUC: 0.8192422772117095

Neural Network (MLP)
Accuracy: 0.75
Precision: 0.7496229260935143
Recall: 0.8367003367003367
F1: 0.7907716785998409
AUC: 0.8162336612118273

Random Forest
Accuracy: 0.7623574144486692
Precision: 0.7559523809523809
Recall: 0.8552188552188552
F1: 0.8025276461295419
AUC: 0.8177499154573391


<br>

### Forward-Looking Results

| Model | Accuracy | Recall | F1 | AUC |
|---|---|---|---|---|
| Logistic Regression | 74.0% | 80.1% | 0.777 | 0.809 |
| XGBoost | 75.6% | 84.0% | 0.795 | 0.819 |
| Neural Network (MLP) | 75.0% | 83.7% | 0.791 | 0.816 |
| **Random Forest** | **76.2%** | **85.5%** | **0.803** | 0.818 |

- **Random Forest is the best model** on accuracy, recall and F1, the same winner as in Part 1.

- **All four models are close**, so the features matter more than the algorithm.

<br>

<br>

Check how much Recency adds, and compare against a simple baseline.

<br>

In [ ]:
# How much does Recency add? Train the same Random Forest without it
cols_without_recency = [c for c in feature_cols_fwd if c != "Recency"]

rf_no_recency = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_no_recency.fit(Xf_train[cols_without_recency], yf_train)
no_recency_preds = rf_no_recency.predict(Xf_test[cols_without_recency])

print("Random Forest without Recency")
print("Accuracy:", accuracy_score(yf_test, no_recency_preds))
print("Recall:", recall_score(yf_test, no_recency_preds))

print("\nBaseline: always predict churn")
print("Accuracy:", yf_test.mean())

Random Forest without Recency
Accuracy: 0.720532319391635
Recall: 0.8535353535353535

Baseline: always predict churn
Accuracy: 0.564638783269962


<br>

- **Recency is the strongest signal.** Without it, accuracy drops from 76.2% to 72.1%.

- **Baseline:** always predicting churn would score 56.5%.

<br>

<br>

---

## Step 13: Is the Result Real?

Two checks before trusting the numbers.

1. **Overfitting check:** does the model score the same on customers it has never seen?

2. **Earlier-period test:** does it still work when trained on an earlier period and tested on a later one?

<br>

<br>

### Check 1: Overfitting

<br>

In [ ]:
# Overfitting check: accuracy on customers the model trained on vs customers it never saw
print("Train accuracy:", accuracy_score(yf_train, rf_fwd.predict(Xf_train)))
print("Test accuracy:", accuracy_score(yf_test, rf_fwd_preds))

Train accuracy: 0.765810746552544
Test accuracy: 0.7623574144486692


<br>

**Result:** no overfitting. Train accuracy (76.6%) and test accuracy (76.2%) are almost the same.

<br>

<br>

### Check 2: Train on an Earlier Period

Build the same dataset again with a cutoff 90 days earlier. These are the same steps as Step 11, with a different date.

<br>

In [ ]:
# Build the same dataset again, with a cutoff 90 days earlier
cutoff_early = cutoff - pd.Timedelta(days=90)

before_early = data[data["InvoiceDate"] <= cutoff_early].copy()
after_early = data[(data["InvoiceDate"] > cutoff_early) & (data["InvoiceDate"] <= cutoff)]

before_early["Amount"] = before_early["Quantity"] * before_early["Price"]

fwd_early = before_early.groupby("Customer ID").agg(
    Frequency=("Invoice", "nunique"),
    Monetary=("Amount", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
)

fwd_early["Tenure"] = (fwd_early["last_purchase"] - fwd_early["first_purchase"]).dt.days
fwd_early["AvgOrderValue"] = fwd_early["Monetary"] / fwd_early["Frequency"]
fwd_early["Frequency_log"] = np.log1p(fwd_early["Frequency"])
fwd_early["Monetary_log"] = np.log1p(fwd_early["Monetary"].clip(lower=0))
fwd_early["Freq_Monetary_Interaction"] = fwd_early["Frequency"] * fwd_early["Monetary"]
fwd_early["Recency"] = (cutoff_early - fwd_early["last_purchase"]).dt.days

bought_again_early = after_early["Customer ID"].unique()
fwd_early["Churned"] = (~fwd_early.index.isin(bought_again_early)).astype(int)

print("Earlier cutoff:", cutoff_early.date())
print("Customers:", len(fwd_early))
print("Churn rate at the earlier cutoff:", fwd_early["Churned"].mean())
print("Churn rate at the main cutoff:", fwd["Churned"].mean())

Earlier cutoff: 2011-06-12
Customers: 4954
Churn rate at the earlier cutoff: 0.6782398062171983
Churn rate at the main cutoff: 0.5644731837200456


<br>

Train the Random Forest on the earlier period, then test it on every customer at the main cutoff.

<br>

In [ ]:
# Train on the earlier period, test on every customer at the main cutoff
rf_early = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_early.fit(fwd_early[feature_cols_fwd], fwd_early["Churned"])

early_preds = rf_early.predict(Xf)
early_probs = rf_early.predict_proba(Xf)[:, 1]

print("Random Forest trained on the earlier period")
print("Accuracy:", accuracy_score(yf, early_preds))
print("Precision:", precision_score(yf, early_preds))
print("Recall:", recall_score(yf, early_preds))
print("AUC:", roc_auc_score(yf, early_probs))

# Simple rule for comparison: flag anyone with no purchase in the last 90 days
rule_preds = (fwd["Recency"] > 90).astype(int)

print("\nSimple rule: no purchase in 90 days")
print("Accuracy:", accuracy_score(yf, rule_preds))
print("Recall:", recall_score(yf, rule_preds))

Random Forest trained on the earlier period
Accuracy: 0.699695701787752
Precision: 0.6660291656705714
Recall: 0.9386792452830188
AUC: 0.7914499935262892

Simple rule: no purchase in 90 days
Accuracy: 0.702738683910232
Recall: 0.8018867924528302


<br>

**Result:** the ranking holds, the yes/no cut-off does not.

- **AUC is 0.79**, close to the 0.82 in Step 12, so the model still ranks customers by risk well on a later period. (AUC of 0.5 is a coin flip, 1.0 is perfect.)

- **Accuracy drops to 70.0%**, level with the simple rule (70.3%).

- **Why:** churn is seasonal. 67.8% of customers went quiet in the summer period, but only 56.4% before Christmas. A model trained on summer flags too many customers in autumn.

**Takeaway:** use the model to **rank customers by churn risk**, and reset the yes/no cut-off for each season.

<br>

<br>

---

## Step 14: The Result in Numbers of Customers

Percentages are hard to picture. This step shows the same results as customer counts.

<br>

<br>

### How Many Leavers the Model Catches

This uses the **1,052 test customers** from Step 12 (the 20% the model never saw during training).

<br>

In [ ]:
from sklearn.metrics import confusion_matrix

# Count the customers on the held-out test set
tn, fp, fn, tp = confusion_matrix(yf_test, rf_fwd_preds).ravel()

print("Test customers:", len(yf_test))
print("Really stopped buying:", tp + fn)
print("Kept buying:", tn + fp)

print("\nRandom Forest")
print("Flagged as leaving:", tp + fp)
print("Correct (really left):", tp)
print("False alarms:", fp)
print("Leavers missed:", fn)

# Same count for the simple rule
rule_test_preds = (Xf_test["Recency"] > 90).astype(int)
tn, fp, fn, tp = confusion_matrix(yf_test, rule_test_preds).ravel()

print("\nSimple rule: no purchase in 90 days")
print("Flagged as leaving:", tp + fp)
print("Correct (really left):", tp)
print("False alarms:", fp)
print("Leavers missed:", fn)

Test customers: 1052
Really stopped buying: 594
Kept buying: 458

Random Forest
Flagged as leaving: 672
Correct (really left): 508
False alarms: 164
Leavers missed: 86

Simple rule: no purchase in 90 days
Flagged as leaving: 685
Correct (really left): 494
False alarms: 191
Leavers missed: 100


<br>

**Result:**

- 594 customers really stopped buying. The model caught **508 of them (85.5%)**.

- It raised 164 false alarms and missed 86 leavers.

- Compared with the simple rule, the model finds **14 more leavers with 27 fewer false alarms**. A real but modest gain.

<br>

<br>

### Ranking Customers by Risk

This uses **all 5,258 customers**. Each one gets a risk score, and they are split into five equal groups of about 1,052 customers each.

<br>

In [ ]:
# Give every customer a risk score, using the model trained on the earlier period
fwd["risk"] = early_probs

# Each customer's spend in the 12 months before the cutoff
last_year = before[before["InvoiceDate"] > cutoff - pd.Timedelta(days=365)]
spend_12m = last_year.groupby("Customer ID")["Amount"].sum()
fwd["spend_12m"] = spend_12m.reindex(fwd.index).fillna(0)

# Five equal groups, from safest to riskiest
fwd["risk_group"] = pd.qcut(
    fwd["risk"].rank(method="first"),
    5,
    labels=["Safest 20%", "Low", "Middle", "High", "Riskiest 20%"],
)

risk_table = fwd.groupby("risk_group", observed=True).agg(
    Customers=("Churned", "size"),
    Stopped_buying=("Churned", "sum"),
    Churn_rate=("Churned", "mean"),
    Spend_12m=("spend_12m", "sum"),
)
risk_table["Share_of_spend"] = risk_table["Spend_12m"] / fwd["spend_12m"].sum()

risk_table.round(3)

,Customers,Stopped_buying,Churn_rate,Spend_12m,Share_of_spend
risk_group,,,,,
Safest 20%,1052,174,0.165,6005182.183,0.702
Low,1051,439,0.418,1257952.802,0.147
Middle,1052,665,0.632,803353.130,0.094
High,1051,787,0.749,380260.730,0.044
Riskiest 20%,1052,903,0.858,103025.710,0.012


<br>

**Result:**

- The ranking works: **85.8% of the riskiest group stopped buying**, against 16.5% of the safest group.

- But the riskiest group brings in only 1.2% of the spend, so it is not where the money is. The next table looks only at the customers who spend the most.

<br>

<br>

### Where the Model Is Most Valuable: High-Value Customers

This is a **smaller group** than the table above. It is narrowed down in two steps:

1. Start with all **5,258** customers.

2. Keep the top 25% by spend in the prior 12 months: **1,315 high-value customers**.

3. Rank those 1,315 by risk score. The riskiest 20% of them is **263 customers**, and the safest 20% is another 263.

<br>

In [ ]:
# High-value customers: the top 25% by spend in the 12 months before the cutoff
high_value = fwd[fwd["spend_12m"] >= fwd["spend_12m"].quantile(0.75)]
high_value = high_value.sort_values("risk", ascending=False)

print("High-value customers:", len(high_value))
print("Share of total spend:", high_value["spend_12m"].sum() / fwd["spend_12m"].sum())
print("Churn rate:", high_value["Churned"].mean())

# Compare the riskiest 20% of them with the safest 20%
k = len(high_value) // 5
riskiest = high_value.head(k)
safest = high_value.tail(k)

print("\nRiskiest 20% of high-value customers")
print("Customers:", len(riskiest))
print("Stopped buying:", riskiest["Churned"].sum())
print("Churn rate:", riskiest["Churned"].mean())
print("Spend in prior 12 months:", round(riskiest["spend_12m"].sum()))

print("\nSafest 20% of high-value customers")
print("Customers:", len(safest))
print("Stopped buying:", safest["Churned"].sum())
print("Churn rate:", safest["Churned"].mean())

High-value customers: 1315
Share of total spend: 0.8259902104518124
Churn rate: 0.21140684410646388

Riskiest 20% of high-value customers
Customers: 263
Stopped buying: 122
Churn rate: 0.46387832699619774
Spend in prior 12 months: 853812

Safest 20% of high-value customers
Customers: 263
Stopped buying: 12
Churn rate: 0.045627376425855515


<br>

**Result:**

- High-value customers bring in **82.6% of the spend**, and 21.1% of them stopped buying.

- The **263 riskiest high-value customers** stopped buying **46.4%** of the time (122 of 263). They spent about £854K in the prior year.

- The **263 safest high-value customers** stopped buying only **4.6%** of the time (12 of 263).

- So among the customers who bring in most of the money, the model's riskiest group is about **10 times more likely to leave** than its safest group.

These 263 customers are the ones worth contacting first.

<br>

<br>

---

## Step 15: The Contact List for the Next 90 Days

Steps 11 to 14 tested the model on a period that had already happened, so the predictions could be checked.

This step uses the model for its real purpose: predicting a period that is **not in the data**. It scores every customer as of the last day of data (2011-12-09) and lists the high-value customers most likely to stop buying in the **following** 90 days.

<br>

<br>

First, train the final model on all the customers from Step 11.

<br>

In [ ]:
# Train the final model on every customer from Step 11 (not just the training split)
final_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
final_model.fit(Xf, yf)

print("Final model trained on", len(Xf), "customers")

Final model trained on 5258 customers


<br>

Next, build each customer's features as of the last day of data. These are the same steps as Step 11, but nothing is held back.

<br>

In [ ]:
# Build the same features again, this time using ALL the data up to its last day
today = data["InvoiceDate"].max()

all_data = data.copy()
all_data["Amount"] = all_data["Quantity"] * all_data["Price"]

current = all_data.groupby("Customer ID").agg(
    Frequency=("Invoice", "nunique"),
    Monetary=("Amount", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
)

current["Tenure"] = (current["last_purchase"] - current["first_purchase"]).dt.days
current["AvgOrderValue"] = current["Monetary"] / current["Frequency"]
current["Frequency_log"] = np.log1p(current["Frequency"])
current["Monetary_log"] = np.log1p(current["Monetary"].clip(lower=0))
current["Freq_Monetary_Interaction"] = current["Frequency"] * current["Monetary"]
current["Recency"] = (today - current["last_purchase"]).dt.days

print("Last day of data:", today.date())
print("Current customers:", len(current))

Last day of data: 2011-12-09
Current customers: 5855


<br>

Give every current customer a risk score.

<br>

In [ ]:
# Risk score for every current customer: chance of no purchase in the next 90 days
current["risk"] = final_model.predict_proba(current[feature_cols_fwd])[:, 1]

# Spend in the last 12 months
recent = all_data[all_data["InvoiceDate"] > today - pd.Timedelta(days=365)]
recent_spend = recent.groupby("Customer ID")["Amount"].sum()
current["spend_12m"] = recent_spend.reindex(current.index).fillna(0)

print("Predicted to stop buying in the next 90 days:", (current["risk"] >= 0.5).sum())
print("Predicted to buy again:", (current["risk"] < 0.5).sum())

Predicted to stop buying in the next 90 days: 3349
Predicted to buy again: 2506


<br>

Narrow it down the same way as Step 14: keep the top 25% by spend, then take the riskiest 20% of those.

<br>

In [ ]:
# High-value customers today: the top 25% by spend in the last 12 months
high_value_now = current[current["spend_12m"] >= current["spend_12m"].quantile(0.75)]
high_value_now = high_value_now.sort_values("risk", ascending=False)

# The riskiest 20% of them is the contact list
k_now = len(high_value_now) // 5
contact_list = high_value_now.head(k_now)

print("High-value customers today:", len(high_value_now))
print("Customers on the contact list:", len(contact_list))
print("Their spend in the last 12 months:", round(contact_list["spend_12m"].sum()))
print("Average risk score on the list:", round(contact_list["risk"].mean(), 3))

High-value customers today: 1464
Customers on the contact list: 292
Their spend in the last 12 months: 882033
Average risk score on the list: 0.485


In [ ]:
# The first 10 customers on the contact list
contact_list[["Frequency", "Recency", "spend_12m", "risk"]].head(10).round(2)

,Frequency,Recency,spend_12m,risk
Customer ID,,,,
16270.0,1,352,1141.24,0.77
12501.0,1,336,2169.39,0.77
12769.0,1,238,1143.60,0.77
15171.0,1,330,1289.50,0.77
12818.0,1,261,1542.08,0.77
12764.0,1,316,1693.88,0.77
13328.0,1,315,1308.48,0.77
16152.0,1,268,1829.04,0.75
13572.0,1,205,1384.25,0.74


<br>

**Result:**

- There are **5,855 current customers**. The model expects 3,349 of them to stop buying in the next 90 days.

- Of the **1,464 high-value customers**, the riskiest 20% are **292 customers**. They spent about **£882K** in the last 12 months.

- **These 292 customers are the list a business would contact first.**

These predictions cannot be checked, because the data ends on 2011-12-09. Step 14 is the evidence for trusting them: when the same method was tested on a past period, 46.4% of the flagged high-value customers really did stop buying.

<br>

<br>

---

## Step 16: MLflow Experiment Tracking

Log every model from Part 1 and Part 2 to MLflow, with its accuracy, precision, recall and F1.

<br>

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn_prediction")

mlflow.set_experiment("churn_prediction")

trusted_types = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer", "sklearn.tree._tree.Tree"]

models_to_log = {
    "LogisticRegression_2features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "XGBoost_2features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "LogisticRegression_7features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "XGBoost_7features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
    "NeuralNetwork_MLP": (
        MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "RandomForest": (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
    # Part 2: the four forward-looking models
    "Forward_LogisticRegression": (
        LogisticRegression(max_iter=1000, random_state=42),
        Xf_train_scaled,
        Xf_test_scaled,
        yf_train,
        yf_test,
    ),
    "Forward_XGBoost": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        Xf_train,
        Xf_test,
        yf_train,
        yf_test,
    ),
    "Forward_NeuralNetwork_MLP": (
        MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        Xf_train_scaled,
        Xf_test_scaled,
        yf_train,
        yf_test,
    ),
    "Forward_RandomForest": (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        Xf_train,
        Xf_test,
        yf_train,
        yf_test,
    ),
}

for name, (model, xtr, xte, ytr, yte) in models_to_log.items():
    with mlflow.start_run(run_name=name):
        model.fit(xtr, ytr)
        preds = model.predict(xte)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", accuracy_score(yte, preds))
        mlflow.log_metric("precision", precision_score(yte, preds))
        mlflow.log_metric("recall", recall_score(yte, preds))
        mlflow.log_metric("f1", f1_score(yte, preds))

        if "XGBoost" in name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model", skops_trusted_types=trusted_types)

print("All models logged to MLflow.")


All models logged to MLflow.


<br>

### MLflow Tracking: Complete

Logged 10 models to MLflow under the "churn_prediction" experiment:

- **Part 1 (snapshot):** Logistic Regression and XGBoost on 2 features, then Logistic Regression, XGBoost, Neural Network and Random Forest on 7 features.

- **Part 2 (forward-looking):** Logistic Regression, XGBoost, Neural Network and Random Forest on 8 features.

Each run stores accuracy, precision, recall and F1, plus the trained model file. Two MLflow security errors came up along the way (untrusted XGBoost and neural network internal types). They were fixed by using XGBoost's dedicated logging function and explicitly trusting the specific flagged types.

<br>

<br>

---

## Step 17: FastAPI Endpoint and Live App

The API (`main.py`) and the live Streamlit app (`streamlit_app.py`) both serve the **forward-looking Random Forest from Step 12**.

- The model is trained once at startup, on all 5,258 customers at the cutoff.

- It takes four inputs: Frequency, Monetary, Tenure and Recency.

- It returns a churn prediction and the probability that the customer stops buying in the next 90 days.

<br>

<br>

### API Testing: Confirmed Working

Tested the /predict endpoint via FastAPI's interactive docs page.

- **Input validation works:** Frequency=0 returns a clean 400 error instead of crashing, and so does a negative Recency.

- **Valid input returns a real prediction**, with a churn label and a probability.

- Example 1: Frequency=10, Monetary=2000, Tenure=400, Recency=20 → 35.2% churn probability, predicted to buy again.

- Example 2: Frequency=1, Monetary=150, Tenure=0, Recency=300 → 87.0% churn probability, predicted to stop buying.

<br>